# 02 — Feature Engineering + Debug Gate

Adds the 3 FE-extension features (stock_to_velocity_weeks, days_since_last_sale_terr, terr_prod_conv_rate), asserts shape/integrity, then fits a default-param CatBoost on the time-split to verify AUC lands in the [0.76, 0.82] gate per CLAUDE.md and the authoritative plan.

In [1]:
import sys, os
from pathlib import Path
_root = Path.cwd()
if _root.name == 'notebooks':
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
os.chdir(_root)

import numpy as np
import pandas as pd

from src.config import (
    DATA_RAW, DATA_PROCESSED, SEED, SPLIT_CUTOFF,
    CAT_LOWCARD, CAT_HIGHCARD, GLOBAL_POS_RATE,
)
from src.utils import seed_all
from src.features.extensions import (
    add_stock_to_velocity_weeks,
    add_days_since_last_sale_terr,
    add_terr_prod_conv_rate,
)
from src.data.split import time_based_split

seed_all()
df = pd.read_parquet(DATA_PROCESSED / 'df_master.parquet')
df['visit_date'] = pd.to_datetime(df['visit_date'])
retailer_pos = pd.read_csv(DATA_RAW / 'retailer_pos.csv')
retailers    = pd.read_csv(DATA_RAW / 'retailers.csv')
print('loaded:', df.shape, '| pos:', retailer_pos.shape, '| retailers:', retailers.shape)

loaded: (30000, 27) | pos: (235042, 7) | retailers: (4000, 5)


In [2]:
df = add_stock_to_velocity_weeks(df)
df[['sku_qty_pre_visit', 'sales_velocity', 'stock_to_velocity_weeks']].describe().round(3)

,sku_qty_pre_visit,sales_velocity,stock_to_velocity_weeks
count,30000.00,30000.000,30000.000
mean,86.91,1.953,109.502
std,47.09,2.376,190.552
min,0.00,0.000,0.000
25%,53.50,0.267,2.875
50%,87.11,1.067,9.725
75%,113.00,2.800,61.837
max,366.00,19.567,500.000


In [3]:
df = add_days_since_last_sale_terr(df, retailer_pos=retailer_pos, retailers=retailers)
print('null after merge:', df['days_since_last_sale_terr'].isnull().sum())
print('value summary:')
print(df['days_since_last_sale_terr'].describe().round(2))
print('\nsentinel (999) count:', (df['days_since_last_sale_terr'] == 999).sum())

null after merge: 0
value summary:
count    30000.00
mean       180.49
std        378.42
min          1.00
25%          2.00
50%          5.00
75%         14.00
max        999.00
Name: days_since_last_sale_terr, dtype: float64

sentinel (999) count: 5282


In [4]:
df = add_terr_prod_conv_rate(df)
print('terr_prod_conv_rate describe:')
print(df['terr_prod_conv_rate'].describe().round(4))
print('\ncorr w/ target:', round(df[['terr_prod_conv_rate','sale_within_7d']].corr().iloc[0,1], 4))

terr_prod_conv_rate describe:
count    30000.0000
mean         0.6590
std          0.3556
min          0.0000
25%          0.5000
50%          0.6667
75%          1.0000
max          1.0000
Name: terr_prod_conv_rate, dtype: float64

corr w/ target: 0.4598


In [5]:
assert df.shape == (30000, 30), f'shape mismatch: {df.shape}'
for c in ['stock_to_velocity_weeks', 'days_since_last_sale_terr', 'terr_prod_conv_rate']:
    assert df[c].isnull().sum() == 0, f'nulls in {c}: {df[c].isnull().sum()}'

conv_mean = df['terr_prod_conv_rate'].mean()
assert abs(conv_mean - GLOBAL_POS_RATE) <= 0.02, f'terr_prod_conv_rate mean off: {conv_mean:.4f}'

n_sentinel = (df['days_since_last_sale_terr'] == 999).sum()
assert 0 < n_sentinel < 30000, f'sentinel count suspicious: {n_sentinel}'

assert df['stock_to_velocity_weeks'].max() <= 500.0 + 1e-6, \
    f'stock_to_velocity_weeks cap broken: {df["stock_to_velocity_weeks"].max()}'

corr = df[['terr_prod_conv_rate','sale_within_7d']].corr().iloc[0,1]
assert corr > 0.20, f'terr_prod_conv_rate corr w/ target too weak: {corr:.4f}'

print('all 5 FE-extension assertions passed')
print(f'  shape:           {df.shape}')
print(f'  conv_mean:       {conv_mean:.4f} (target ~{GLOBAL_POS_RATE})')
print(f'  sentinel(999):   {n_sentinel}')
print(f'  stk/vel max:     {df["stock_to_velocity_weeks"].max():.2f}')
print(f'  corr w/ target:  {corr:.4f}')

all 5 FE-extension assertions passed
  shape:           (30000, 30)
  conv_mean:       0.6590 (target ~0.651)
  sentinel(999):   5282
  stk/vel max:     500.00
  corr w/ target:  0.4598


In [6]:
out = DATA_PROCESSED / 'df_master_extended.parquet'
df.to_parquet(out, index=False)
print('saved:', out, df.shape)

saved: data\processed\df_master_extended.parquet (30000, 30)


In [7]:
# DEBUG GATE — default-param CatBoost on time-split. AUC must land in [0.76, 0.82].
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

# Plan formula: CAT_LOWCARD (4) + CAT_HIGHCARD (3) + NUM_COLS (13) + BOOL_COLS (5) + FE_EXT (3) = 28.
# Plan's parenthetical "25" omits CAT_HIGHCARD identifiers in the count but the formula includes
# them as predictors — that's the configuration that produced the 0.7820 prior-run AUC.
FEATURE_COLS = (
    list(CAT_LOWCARD) + list(CAT_HIGHCARD) +
    ['week_of_season','month','day_of_week',
     'days_since_last_visit','visit_count_last_30days','days_since_product_last_pushed',
     'sku_qty_pre_visit','stock_change_wow',
     'rolling_avg_weekly_sales','sales_velocity','total_revenue_last_30d',
     'rep_visit_frequency_score','rep_product_diversity'] +
    ['is_critical_period','is_harvest_approaching','is_first_visit_for_product',
     'is_out_of_stock','product_sold_last_14d'] +
    ['stock_to_velocity_weeks','days_since_last_sale_terr','terr_prod_conv_rate']
)
assert len(FEATURE_COLS) == 28, f'feature count off: {len(FEATURE_COLS)}'

cat_features = list(CAT_LOWCARD) + list(CAT_HIGHCARD)
X = df[FEATURE_COLS].copy()
for c in cat_features:
    X[c] = X[c].astype(str)
y = df['sale_within_7d'].values

train_idx, test_idx = time_based_split(df, cutoff=SPLIT_CUTOFF)
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y[train_idx], y[test_idx]
print('train:', X_tr.shape, '| test:', X_te.shape)

model = CatBoostClassifier(random_state=SEED, verbose=0)
model.fit(X_tr, y_tr, cat_features=cat_features)
auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

print(f'\nCatBoost default-param test AUC = {auc:.4f}')
if 0.76 <= auc <= 0.82:
    print(f'floor confirmed: AUC={auc:.4f}')
elif auc < 0.76:
    raise AssertionError(
        f'DEBUG GATE FAILED (AUC={auc:.4f} < 0.76). Check (in order): '
        'missing shift(1) in terr_prod_conv_rate, cat_features not passed, '
        'wrong sentinel for days_since_last_sale_terr, allow_exact_matches=True in any merge_asof.'
    )
else:
    raise AssertionError(
        f'DEBUG GATE FAILED (AUC={auc:.4f} >= 0.82). Possible leakage in 3 FE extensions; '
        'verify shift(1), allow_exact_matches=False, and that no post-visit_date data leaks in.'
    )

train: (23862, 28) | test: (6138, 28)



CatBoost default-param test AUC = 0.7698
floor confirmed: AUC=0.7698
